# MuonClip angular/radial RG analysis — plots first

This notebook compares the exact saved **initial**, **best**, and **final** checkpoints for one transformer matrix, defaulting to `L00_W_Q`.

The decomposition is

$$
W_t = U_t \Sigma_t V_t^\top .
$$

The ordinary WeightWatcher ESD uses

$$
\lambda_i(t)=\sigma_i(t)^2,
$$

so it is a **radial/singular-value observable**.

To remove that radial sector, replace every singular value by one:

$$
Q_t = U_t V_t^\top .
$$

For a square Gaussian random matrix, this polar factor is Haar-distributed on $O(N)$. For a rectangular matrix the corresponding object lies on a Stiefel manifold.

We therefore compare the trained angular flow against a matched Haar/Stiefel null. The hypothesis being tested is whether the projective angular spectrum develops a non-random heavy tail whose fitted exponent approaches the marginal value

$$
\alpha = 2.
$$

**This notebook always displays the generated plots inline.** The analysis engine saves figures to disk with a noninteractive backend; after the analysis finishes, the notebook explicitly embeds every PNG. `ANGULAR_SHOW_PLOTS` is retained as a Papermill compatibility input, but it does not disable the inline PNG summary.


## What is plotted

For the selected matrix the notebook shows:

1. radial spectra for initial, best, and final checkpoints;
2. angular `tilt` and `twist` power-law fits;
3. linear-scale angular PDFs;
4. CDF and CCDF plots;
5. far-tail CCDF comparisons against the Haar/Stiefel null ensemble;
6. pairwise $\alpha$ values versus the random-angular 95% interval, with $\alpha=2$ marked;
7. pairwise tail length, KS $D$, and package-selected $x_{\min}$ versus the random null.

The power-law fit receives **all positive continuous projective values**. We supply no `xmin` and no `xmax`; `powerlaw.Fit` chooses `xmin` by its MLE/KS procedure. Endpoint atoms are counted separately instead of being mapped to enormous artificial projective values.


In [ ]:
import os

TARGET_SEED = int(os.environ.get("TARGET_SEED", os.environ.get("RG_SEED", "4242")))
TARGET_OPTIMIZER = os.environ.get("TARGET_OPTIMIZER", os.environ.get("OPTIMIZER_NAME", "muon_clip"))
RUNROOT = os.environ.get("RUNROOT", "")
RESULTS_ROOT = os.environ.get("RESULTS_ROOT", "")
RUN_DIR = os.environ.get("RUN_DIR", "")
INITIAL_CHECKPOINT_PATH = os.environ.get("INITIAL_CHECKPOINT_PATH", "")
BEST_CHECKPOINT_PATH = os.environ.get("BEST_CHECKPOINT_PATH", "")
FINAL_CHECKPOINT_PATH = os.environ.get("FINAL_CHECKPOINT_PATH", os.environ.get("CHECKPOINT_PATH", ""))
RG_MATRIX_NAME = os.environ.get("RG_MATRIX_NAME", "L00_W_Q")

# Keep this moderate for the first run. Increase to 100 or 500 later.
ANGULAR_N_NULL = int(os.environ.get("ANGULAR_N_NULL", "32"))
ANGULAR_MIN_TAIL = int(os.environ.get("ANGULAR_MIN_TAIL", "20"))
ANGULAR_NULL_SEED = int(os.environ.get("ANGULAR_NULL_SEED", "91337"))
ANGULAR_ENDPOINT_TOL = float(os.environ.get("ANGULAR_ENDPOINT_TOL", "1e-10"))
ANGULAR_SHOW_PLOTS = os.environ.get("ANGULAR_SHOW_PLOTS", "0")


In [ ]:
from pathlib import Path
from contextlib import contextmanager
import gc
import sys

# Analysis writes PNGs using a noninteractive backend. We display the saved
# PNGs ourselves afterward, so plots are always visible in the notebook.
os.environ["MPLBACKEND"] = "Agg"
os.environ["ANGULAR_ENDPOINT_TOL"] = str(ANGULAR_ENDPOINT_TOL)
if str(BEST_CHECKPOINT_PATH).strip():
    os.environ["BEST_CHECKPOINT_PATH"] = str(BEST_CHECKPOINT_PATH)
else:
    os.environ.pop("BEST_CHECKPOINT_PATH", None)

def _none_if_blank(value):
    text = str(value).strip() if value is not None else ""
    return text or None

def find_experiment_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = []
    root_env = os.environ.get("RG_OPTIMIZERS_ROOT")
    if root_env:
        root = Path(root_env).expanduser().resolve()
        candidates += [root, root / "baseline" / "nanogpt_one_head"]
    for base in (cwd, *cwd.parents):
        candidates += [base, base / "baseline" / "nanogpt_one_head"]
    for candidate in dict.fromkeys(candidates):
        if (candidate / "src" / "rg_nanogpt_one_head" / "model.py").is_file():
            return candidate
    raise FileNotFoundError("Launch from rg_optimizers or set RG_OPTIMIZERS_ROOT.")

EXPERIMENT_ROOT = find_experiment_root()
sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))

from rg_nanogpt_one_head.angular_weightwatcher_core import AnalysisConfig
import rg_nanogpt_one_head.angular_three_checkpoint as angular_three

CONFIG = AnalysisConfig(
    seed=int(TARGET_SEED),
    optimizer=str(TARGET_OPTIMIZER).lower(),
    runroot=_none_if_blank(RUNROOT),
    results_root=_none_if_blank(RESULTS_ROOT),
    run_dir=_none_if_blank(RUN_DIR),
    initial_checkpoint=_none_if_blank(INITIAL_CHECKPOINT_PATH),
    final_checkpoint=_none_if_blank(FINAL_CHECKPOINT_PATH),
    angular_nulls=ANGULAR_N_NULL,
    min_tail=ANGULAR_MIN_TAIL,
    null_seed=ANGULAR_NULL_SEED,

    # IMPORTANT: keep matplotlib itself headless while computing. The next
    # cells embed every saved PNG into the notebook explicitly.
    show_plots=False,
)

print("RUNROOT       =", RUNROOT or "<unset>")
print("RESULTS_ROOT  =", RESULTS_ROOT or "<unset>")
print("RUN_DIR       =", RUN_DIR or "<resolved from roots>")
print("MATRIX         =", RG_MATRIX_NAME)
print("NULLS          =", ANGULAR_N_NULL)
print("MIN_TAIL       =", ANGULAR_MIN_TAIL)
print("PLOTS INLINE   = ALWAYS (saved PNGs are embedded below)")
print("SHOW_PLOTS     =", ANGULAR_SHOW_PLOTS, "(compatibility input; inline PNGs remain enabled)")


## Run the single-matrix angular analysis

The shared engine knows how to analyze all six transformer matrices. For this notebook we filter the checkpoint matrices **before** generating null ensembles or power-law fits, so only `RG_MATRIX_NAME` is analyzed. This keeps the workload small without changing the scientific calculation for that matrix.


In [ ]:
@contextmanager
def analyze_only_matrix(matrix_name: str):
    original_loader = angular_three._load_three_weight_sets

    def filtered_loader(config, resolved, paths):
        weights, payloads, model_cfg = original_loader(config, resolved, paths)
        available = sorted(weights["initial"])
        if matrix_name not in available:
            raise KeyError(
                f"{matrix_name!r} unavailable. Available matrices: {available}"
            )
        filtered = {
            state: {matrix_name: matrices[matrix_name]}
            for state, matrices in weights.items()
        }
        return filtered, payloads, model_cfg

    angular_three._load_three_weight_sets = filtered_loader
    try:
        yield
    finally:
        angular_three._load_three_weight_sets = original_loader
        gc.collect()

with analyze_only_matrix(RG_MATRIX_NAME):
    RESULTS, MANIFEST = angular_three.run_three_checkpoint_analysis(CONFIG)

OUTPUT_DIR = Path(MANIFEST["output_dir"])

print("\nCHECKPOINTS")
for state, path in MANIFEST["checkpoints"].items():
    print(f"{state:7s} step={MANIFEST['steps'][state]:7d}  {path}")

print("\nOUTPUT_DIR =", OUTPUT_DIR)
print("PNG count  =", len(list(OUTPUT_DIR.glob("*.png"))))


# Plots

The cells below embed the saved PNG files directly. This is intentional: it avoids keeping dozens of Matplotlib figures alive while guaranteeing that the plots are visible in Jupyter and in the executed Papermill notebook.


In [ ]:
from IPython.display import display, Image, Markdown

pngs = sorted(OUTPUT_DIR.glob("*.png"))
if not pngs:
    raise RuntimeError(f"No PNG plots were generated in {OUTPUT_DIR}")

# Put the most important diagnostics first.
def priority(path):
    name = path.name
    groups = [
        "radial_initial_best_final",
        "pairwise_alpha_vs_random",
        "pairwise_tail_decades_vs_random",
        "initial_to_final",
        "pairwise_powerlaw_D_vs_random",
        "pairwise_xmin_vs_random",
    ]
    for i, token in enumerate(groups):
        if token in name:
            return (i, name)
    return (len(groups), name)

pngs = sorted(pngs, key=priority)

for path in pngs:
    display(Markdown(f"### `{path.name}`"))
    display(Image(filename=str(path), width=1000))


# Numerical summary

A missing `actual_alpha` is **not** silently ignored. It means the package-selected tail did not satisfy the acceptance requirements—most commonly because fewer than `ANGULAR_MIN_TAIL` observations remained above the package-selected $x_{\min}$, or because the checkpoint pair is effectively identical.

The table below exposes the fit-success flag, tail population, tail extent, and random-null interval next to $\alpha$ so the result is interpretable.


In [ ]:
from IPython.display import display
import pandas as pd
import numpy as np

cols = [
    "pair",
    "angular_type",
    "pair_identical_weights",
    "actual_fit_success",
    "actual_alpha",
    "null_alpha_2p5",
    "null_alpha_median",
    "null_alpha_97p5",
    "actual_xmin",
    "actual_D",
    "actual_tail_n",
    "actual_tail_decades",
    "actual_endpoint_atoms",
    "actual_zero_atoms",
    "full_continuous_ks_mc_p",
    "tail_conditional_ks_mc_p",
    "candidate_nonrandom_long_tail",
]
display(RESULTS)
summary = RESULTS[cols].copy()
display(summary)

failed = summary[~summary["actual_fit_success"].astype(bool)]
if len(failed):
    print("\nFAILED / UNACCEPTED POWER-LAW FITS")
    display(
        failed[
            [
                "pair",
                "angular_type",
                "pair_identical_weights",
                "actual_tail_n",
                "actual_tail_decades",
                "actual_endpoint_atoms",
                "actual_zero_atoms",
            ]
        ]
    )


# RG interpretation

The claim we want to test is stronger than merely observing $\alpha \approx 2$.

Evidence for a non-random marginal angular flow should simultaneously show:

$$
\alpha_{\mathrm{actual}} \rightarrow 2,
$$

a nontrivial fitted tail with adequate population and dynamic range, and statistically meaningful separation from the matched Haar/Stiefel null.

Conversely, if the radial ESD remains close to its random baseline while these angular observables separate strongly from Haar, that supports the hypothesis that MuonClip learns primarily through singular-vector organization while keeping the singular-value bulk comparatively randomized.


In [ ]:
# Concise verdict helper: this does not replace inspecting the plots.
verdict = RESULTS[
    [
        "pair",
        "angular_type",
        "actual_fit_success",
        "actual_alpha",
        "actual_tail_n",
        "actual_tail_decades",
        "full_continuous_ks_mc_p",
        "tail_conditional_ks_mc_p",
        "candidate_nonrandom_long_tail",
    ]
].copy()

verdict["distance_to_alpha_2"] = np.abs(verdict["actual_alpha"] - 2.0)
display(verdict.sort_values(["pair", "angular_type"]))

print("\nSummary CSV:", MANIFEST["summary_csv"])
print("All plots:   ", OUTPUT_DIR)
